# Treino culinário — projeto tinyLLM

Usar os três ficheiros produzidos por `preparar.py treino`. Ver `NOVO_TREINO.md`. O código de treino mantém os parâmetros da experiência v02; os caminhos de entrada são agora genéricos. A validação deve ser a mesma da experiência anterior.

# TinySLM — segundo SFT culinário v0.2
Adaptação do notebook Chat MVP fornecido. Treino supervisionado com receita + pergunta/histórico como entrada e resposta do assistente como alvo.

**No Kaggle:** selecionar GPU; adicionar o dataset do modelo `473_B1`, o tokenizer chat original e os ficheiros `train.jsonl`, `validation.jsonl`, `dataset_report.json` deste pacote. Não é necessário adicionar OASST1. Executar de cima para baixo.

Por defeito parte do **473_B1 (~51,3M parâmetros, segundo o notebook original)**. Este ficheiro não contém pesos nem resultados de treino. Para partir de outro checkpoint, definir `BASE_OVERRIDE` e confirmar que o tokenizer é o mesmo. A avaliação abaixo mede generalização SFT a receitas reservadas; se a base já viu estas receitas noutro treino, não constitui um teste de receitas totalmente inéditas.

Mantemos 512 tokens, 3 epochs e LR 5e-5 como experiência inicial. Não juntamos conversas em blocos arbitrários. Cada resposta é treinada com a receita e todo o histórico anterior. Exemplos longos são excluídos e contabilizados, nunca truncados. A loss incide apenas na resposta atual e no EOS.

## Comparação com v01
Recomeçar nos mesmos pesos **473_B1**, não na epoch 3 anterior. Mantemos tokenizer, prompt, 512 tokens, LR, batch, seed e 3 epochs. A validação é exatamente o mesmo ficheiro (372 conversas de 13 receitas); novas conversas dessas receitas ficam reservadas fora do treino. A seleção das 14 gerações permanece igual se o tokenizer for o mesmo.

Com mais dados e o mesmo número de epochs haverá mais atualizações e mais tokens processados. Esta experiência mede a expansão do treino; não isola diversidade com orçamento computacional fixo. A cobertura efetiva dos 512 tokens será medida durante a execução.


In [ ]:
import os, sys, json, math, random, time, hashlib, collections, shutil
from pathlib import Path
import numpy as np
import torch
import transformers
from transformers import AutoTokenizer, LlamaForCausalLM
from torch.utils.data import DataLoader
from torch.optim import AdamW
print("Python", sys.version.split()[0], "Torch", torch.__version__, "Transformers", transformers.__version__)
assert torch.cuda.is_available(), "Ative uma GPU nas definições do notebook Kaggle."
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
SEED = 42
MAX_LEN = 512
EPOCHS = 3
LR = 5e-5
MICRO_BATCH = 4
GRAD_ACCUM = 4
MAX_NEW_TOKENS = 128
# Se a descoberta automática não for única, indicar as pastas exatas abaixo.
BASE_OVERRIDE = ""
TOKENIZER_OVERRIDE = ""
DATA_OVERRIDE = ""  # pasta com train.jsonl e validation.jsonl deste pacote
RUN_NAME = "culinario_v03"  # mudar em cada experiência
OUT = Path("/kaggle/working") / RUN_NAME
OUT.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
files = [p for p in Path("/kaggle/input").rglob("*") if p.is_file()]
def unique(name, contains):
    hits = [p for p in files if p.name == name and contains in str(p)]
    if len(hits) != 1:
        raise RuntimeError(f"Defina o caminho manualmente: {name}, {contains}: {hits}")
    return hits[0]
BASE_DIR = Path(BASE_OVERRIDE) if BASE_OVERRIDE else unique("config.json", "473_B1").parent
TOKENIZER_DIR = Path(TOKENIZER_OVERRIDE) if TOKENIZER_OVERRIDE else unique("tokenizer.json", "tinyslm-tokenizer-ingles-bpe8k-v02-chat").parent
DATA_DIR = Path(DATA_OVERRIDE) if DATA_OVERRIDE else unique("dataset_report.json", "").parent
print("Base:", BASE_DIR, "\nTokenizer:", TOKENIZER_DIR, "\nDados:", DATA_DIR)
def read_jsonl(path):
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
train_rows = read_jsonl(DATA_DIR / "train.jsonl")
val_rows = read_jsonl(DATA_DIR / "validation.jsonl")
assert not {r['recipe_id'] for r in train_rows} & {r['recipe_id'] for r in val_rows}
print("Conversas:", len(train_rows), "treino /", len(val_rows), "validação")

report = json.loads((DATA_DIR / "dataset_report.json").read_text())
assert hashlib.sha256((DATA_DIR / "validation.jsonl").read_bytes()).hexdigest() == report["validation_sha256"]
assert len(train_rows) == report["train_examples"]
assert len(val_rows) == report["validation_examples"]


In [ ]:
tok = AutoTokenizer.from_pretrained(TOKENIZER_DIR, local_files_only=True)
assert len(tok) == 8005, "Use o tokenizer chat BPE8k original."
assert tok.convert_tokens_to_ids("<|user|>") == 8000
assert tok.convert_tokens_to_ids("<|assistant|>") == 8001
assert tok.bos_token_id is not None and tok.eos_token_id is not None
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token
model = LlamaForCausalLM.from_pretrained(BASE_DIR, local_files_only=True, torch_dtype=torch.float32)
assert MAX_LEN <= model.config.max_position_embeddings, "Contexto superior ao configurado na base."
old_vocab = model.get_input_embeddings().weight.shape[0]
assert old_vocab in (8000, 8005)
if old_vocab == 8000:
    old_in = model.get_input_embeddings().weight.detach().clone()
    old_out = model.get_output_embeddings().weight.detach().clone()
    model.resize_token_embeddings(len(tok))
    with torch.no_grad():
        # Inicializar também a cabeça de saída quando não partilha os pesos.
        pairs = [(model.get_input_embeddings().weight, old_in)]
        if model.get_output_embeddings().weight.data_ptr() != model.get_input_embeddings().weight.data_ptr():
            pairs.append((model.get_output_embeddings().weight, old_out))
        for weight, old in pairs:
            weight[8000:].copy_(old.mean(0) + torch.randn_like(weight[8000:]) * old.std() * .01)
        assert torch.equal(model.get_input_embeddings().weight[:8000], old_in)
        assert torch.equal(model.get_output_embeddings().weight[:8000], old_out)
    del old_in, old_out
model.config.pad_token_id = tok.pad_token_id
model.config.eos_token_id = tok.eos_token_id
model.config.bos_token_id = tok.bos_token_id
device = torch.device("cuda")
model.to(device)
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
scaler = torch.amp.GradScaler("cuda", enabled=AMP_DTYPE == torch.float16)
print("Parâmetros:", sum(p.numel() for p in model.parameters()), "Precisão:", AMP_DTYPE)


## Preparação e cobertura do contexto
O modelo vê a receita em todos os turnos. Num segundo turno, vê também a primeira pergunta e resposta. Estas respostas anteriores são contexto, não alvos duplicados. Não recebe códigos de persona/capacidade nem as evidências como texto de treino.

O relatório mostra percentis de comprimento, percentagem de respostas que cabem e cobertura por capacidade. Se poucas couberem, esta experiência mede sobretudo as receitas curtas. Não aumentar automaticamente o contexto sem avaliar a base.


In [ ]:
SYSTEM = "Answer in English using the supplied recipe. Follow the user's request. If the recipe does not establish an answer, state the missing or conflicting information."
def enc(text):
    return tok.encode(text, add_special_tokens=False)
def prompt_ids(row, assistant_index):
    ids = [tok.bos_token_id] + enc("<|system|>\n" + SYSTEM + "\n")
    context = "\n\n".join(c["text"] for c in row["context"]["chunks"])
    for i, message in enumerate(row["conversation"][:assistant_index]):
        content = message["content"]
        if i == 0:
            content = "Recipe context:\n" + context + "\n\nQuestion:\n" + content
        ids += enc("<|" + message["role"] + "|>\n" + content)
        if message["role"] == "assistant":
            ids += [tok.eos_token_id]
        ids += enc("\n")
    return ids + enc("<|assistant|>\n")
def prepare(rows, split):
    samples, audit = [], []
    for row in rows:
        for i, message in enumerate(row["conversation"]):
            if message["role"] != "assistant":
                continue
            prompt = prompt_ids(row, i)
            target = enc(message["content"]) + [tok.eos_token_id]
            ids = prompt + target
            keep = len(ids) <= MAX_LEN
            audit.append({"id":row["id"], "turn":i//2+1, "capability":row["capability"], "tokens":len(ids), "kept":keep})
            if keep:
                samples.append({"input_ids":ids, "labels":[-100]*len(prompt)+target,
                                "row":row, "assistant_index":i, "prompt":prompt})
    (OUT / f"length_audit_{split}.json").write_text(json.dumps(audit, indent=2))
    counts = collections.Counter(x['capability'] for x in audit if x['kept'])
    lengths = [x['tokens'] for x in audit]
    print(split, "respostas:",len(audit), "cabem:",len(samples), f"({len(samples)/len(audit):.1%})")
    print("Tokens p50/p90/p95/max:", np.percentile(lengths,[50,90,95,100]).astype(int))
    print("Respostas mantidas por capacidade:",dict(counts))
    if len(samples) < .5*len(audit):
        print("ATENÇÃO: menos de metade cabe; o teste representa o subconjunto curto.")
    assert samples, "Nenhuma resposta cabe no contexto. Não iniciar o treino."
    return samples
train_samples = prepare(train_rows, "train")
val_samples = prepare(val_rows, "validation")
def collate(samples):
    size = max(len(s['input_ids']) for s in samples)
    ids = torch.full((len(samples),size),tok.pad_token_id,dtype=torch.long)
    labels = torch.full_like(ids,-100)
    mask = torch.zeros_like(ids)
    for i,s in enumerate(samples):
        n=len(s['input_ids'])
        ids[i,:n]=torch.tensor(s['input_ids']); labels[i,:n]=torch.tensor(s['labels']); mask[i,:n]=1
    return {"input_ids":ids,"labels":labels,"attention_mask":mask}
train_loader = DataLoader(train_samples, batch_size=MICRO_BATCH, shuffle=True,
                          generator=torch.Generator().manual_seed(SEED), collate_fn=collate, num_workers=0)
val_loader = DataLoader(val_samples, batch_size=MICRO_BATCH, collate_fn=collate, num_workers=0)
print("Batches/epoch:",len(train_loader))


## Avaliação fixa: antes e depois
Loss/perplexidade são calculadas apenas sobre respostas do assistente, ponderadas pelo número de tokens. Não comparar diretamente com a loss antiga de texto completo.

Selecionamos até dois casos por capacidade que permitam reservar 128 tokens para geração. As mesmas perguntas são usadas em todas as epochs, com geração greedy. Nos turnos seguintes usamos o histórico de referência (teacher-forced history); isto não mede uma conversa inteiramente gerada pelo modelo. As respostas de referência são guias, não a única formulação aceitável.


In [ ]:
@torch.no_grad()
def evaluate():
    model.eval(); total_loss=0.; total_tokens=0
    for batch in val_loader:
        batch={k:v.to(device) for k,v in batch.items()}
        n=int((batch['labels'][:,1:] != -100).sum())
        with torch.autocast(device_type="cuda",dtype=AMP_DTYPE):
            loss=model(**batch,use_cache=False).loss
        if not torch.isfinite(loss): raise RuntimeError("Loss de validação não finita")
        total_loss+=float(loss)*n; total_tokens+=n
    result=total_loss/total_tokens
    return {"validation_loss":result,"assistant_perplexity":math.exp(min(result,20)),"target_tokens":total_tokens}
eval_cases=[]
for cap in sorted({s['row']['capability'] for s in val_samples}):
    eligible=[s for s in val_samples if s['row']['capability']==cap and len(s['prompt'])+MAX_NEW_TOKENS<=MAX_LEN]
    # Priorizar follow-up em C6; impedir que os dois exemplos sejam a mesma conversa.
    eligible.sort(key=lambda s:(-s['assistant_index'] if cap=='C6' else 0, s['row']['id']))
    used=set()
    for s in eligible:
        if s['row']['id'] in used: continue
        eval_cases.append(s); used.add(s['row']['id'])
        if len(used)==2: break
    print(cap, "casos de geração:",len(used))
assert eval_cases, "Nenhum caso permite gerar 128 tokens com contexto completo."
@torch.no_grad()
def generation_suite(stage):
    model.eval(); results=[]
    stops=list(set([tok.eos_token_id]+[tok.convert_tokens_to_ids(t) for t in ['<|user|>','<|assistant|>','<|system|>','<|tool|>','<|tool_result|>']]))
    for s in eval_cases:
        x=torch.tensor([s['prompt']],device=device)
        with torch.autocast(device_type="cuda",dtype=AMP_DTYPE):
            y=model.generate(input_ids=x,attention_mask=torch.ones_like(x),max_new_tokens=MAX_NEW_TOKENS,
                             do_sample=False,eos_token_id=stops,pad_token_id=tok.pad_token_id)
        answer=tok.decode(y[0,len(s['prompt']):],skip_special_tokens=True)
        row=s['row']; idx=s['assistant_index']
        results.append({"stage":stage,"id":row['id'],"capability":row['capability'],"persona":row['persona'],
                        "recipe_id":row['recipe_id'],"context":row['context'],"history":row['conversation'][:idx],
                        "reference":row['conversation'][idx]['content'],"prediction":answer,
                        "generated_tokens":len(y[0])-len(s['prompt']),"stopped":int(y[0,-1]) in stops})
        print(stage,row['capability'],row['conversation'][idx-1]['content'],"\n=>",answer,"\n")
    return results
history=[{"stage":"base","epoch":0,**evaluate()}]
generations=generation_suite("base")
def persist():
    (OUT/'metrics.json').write_text(json.dumps(history,indent=2))
    (OUT/'generations.json').write_text(json.dumps(generations,ensure_ascii=False,indent=2))
    (OUT/'run_config.json').write_text(json.dumps({"seed":SEED,"max_len":MAX_LEN,"lr":LR,"epochs":EPOCHS,
        "base":str(BASE_DIR),"tokenizer":str(TOKENIZER_DIR),"torch":torch.__version__,"transformers":transformers.__version__,
        "train_sha256":hashlib.sha256((DATA_DIR/'train.jsonl').read_bytes()).hexdigest(),
        "validation_sha256":hashlib.sha256((DATA_DIR/'validation.jsonl').read_bytes()).hexdigest()},indent=2))
persist()
print(history[0])


In [ ]:
optimizer=AdamW(model.parameters(),lr=LR,betas=(.9,.95),weight_decay=.1)
total_steps=math.ceil(len(train_loader)/GRAD_ACCUM)*EPOCHS
warmup=max(1,int(.05*total_steps))
scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lambda step:min(1.,(step+1)/warmup))
best_loss=float('inf')
for epoch in range(1,EPOCHS+1):
    model.train(); started=time.time(); total_nll=0.; total_n=0; group_n=0
    optimizer.zero_grad(set_to_none=True)
    for index,batch in enumerate(train_loader,1):
        batch={k:v.to(device) for k,v in batch.items()}
        n=int((batch['labels'][:,1:]!=-100).sum())
        with torch.autocast(device_type='cuda',dtype=AMP_DTYPE):
            loss=model(**batch,use_cache=False).loss
        if not torch.isfinite(loss): raise RuntimeError("Loss não finita; parar e inspecionar.")
        # Acumular somas de NLL e dividir pelo total real de tokens do grupo.
        # A escala fixa reduz a magnitude no backward sem alterar a média final.
        scale_tokens=MICRO_BATCH*MAX_LEN*GRAD_ACCUM
        scaler.scale(loss*n/scale_tokens).backward()
        total_nll+=float(loss.detach())*n; total_n+=n; group_n+=n
        if index%GRAD_ACCUM==0 or index==len(train_loader):
            scaler.unscale_(optimizer)
            for parameter in model.parameters():
                if parameter.grad is not None: parameter.grad.mul_(scale_tokens/group_n)
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
            old_scale=scaler.get_scale()
            scaler.step(optimizer); scaler.update()
            if scaler.get_scale()>=old_scale: scheduler.step()
            optimizer.zero_grad(set_to_none=True); group_n=0
        if index%50==0 or index==len(train_loader):
            print(f"Epoch {epoch} | {index}/{len(train_loader)} | loss {total_nll/total_n:.4f}")
    metrics={"stage":f"epoch_{epoch}","epoch":epoch,"train_loss":total_nll/total_n,
             "minutes":(time.time()-started)/60,**evaluate()}
    history.append(metrics)
    checkpoint=OUT/f"epoch_{epoch}"
    model.save_pretrained(checkpoint,safe_serialization=True); tok.save_pretrained(checkpoint)
    if metrics['validation_loss']<best_loss:
        best_loss=metrics['validation_loss']
        (OUT/'best_checkpoint.json').write_text(json.dumps({"checkpoint":checkpoint.name,**metrics},indent=2))
    generations+=generation_suite(f"epoch_{epoch}")
    persist(); print(metrics)
# Checkpoints de pesos; não incluem optimizer/RNG para retoma exata.


In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(7,3))
plt.plot([m['epoch'] for m in history],[m['validation_loss'] for m in history],marker='o',label='Validação: respostas')
plt.xlabel('Epoch (0 = base)'); plt.ylabel('Loss'); plt.grid(alpha=.25); plt.legend(); plt.show()
# CSV de avaliação manual: 0=incorreto, 1=parcial, 2=correto. Não preenchido automaticamente.
import csv
with (OUT/'manual_review.csv').open('w',newline='') as f:
    fields=['stage','id','capability','reference','prediction','correct_0_2','follows_request_0_2','grounded_0_2','notes']
    writer=csv.DictWriter(f,fieldnames=fields);writer.writeheader()
    for r in generations: writer.writerow({k:r.get(k,'') for k in fields})
zip_path=shutil.make_archive('/kaggle/working/TinySLM_Culinary_PoC_v02_results','zip',OUT)
print('Resultados:',zip_path)
print('Melhor checkpoint SFT por validation loss:',(OUT/'best_checkpoint.json').read_text())


## O que enviar para analisar
Descarregar `metrics.json`, `generations.json`, `length_audit_train.json`, `length_audit_validation.json` e `run_config.json`, ou o ZIP final com os checkpoints.

O teste é promissor se as respostas melhorarem face à base, seguirem os pedidos e não inventarem detalhes. Uma loss menor não basta. Rever especialmente C4 (restrições), C6 (contexto) e C7 (lacunas). O modelo pequeno pode precisar de mais treino linguístico ou de dados diferentes; os erros não podem ser atribuídos automaticamente apenas ao dataset.

Limitações: validação pequena e usada para escolher a epoch (não é teste final independente); ausência de revisão semântica exaustiva; só exemplos que cabem em 512 tokens; não avalia conhecimento culinário externo; não inclui notas E4 neste snapshot. Fixar este split para comparar versões. Uma futura versão de dados deve preservar as receitas reservadas.

Referência de implementação AMP: https://docs.pytorch.org/docs/stable/notes/amp_examples.html (GradScaler, unscale antes de clipping e acumulação).
